<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#유튜브-레시피-추출-강화-루프" data-toc-modified-id="유튜브-레시피-추출-강화-루프-1">유튜브 레시피 추출 강화 루프</a></span><ul class="toc-item"><li><span><a href="#1.-환경-점검-&amp;-모듈-로드" data-toc-modified-id="1.-환경-점검-&amp;-모듈-로드-1.1">1. 환경 점검 &amp; 모듈 로드</a></span></li><li><span><a href="#2.-설정-&amp;-기준선-확인" data-toc-modified-id="2.-설정-&amp;-기준선-확인-1.2">2. 설정 &amp; 기준선 확인</a></span></li><li><span><a href="#3.-배선-스모크-(쿼터/구현-없이)" data-toc-modified-id="3.-배선-스모크-(쿼터/구현-없이)-1.3">3. 배선 스모크 (쿼터/구현 없이)</a></span></li><li><span><a href="#4.-한-ITER-실행-(선택)" data-toc-modified-id="4.-한-ITER-실행-(선택)-1.4">4. 한 ITER 실행 (선택)</a></span></li><li><span><a href="#5.-전체-루프-실행-(선택)" data-toc-modified-id="5.-전체-루프-실행-(선택)-1.5">5. 전체 루프 실행 (선택)</a></span></li></ul></li></ul></div>

# 유튜브 레시피 추출 강화 루프

`계획(Claude) → 구현(Codex) → 확정검증 → 채점(결정적+AI) → 판정 → 진단 → 재시도` 루프로
`lib/server/recipe-extraction-lab/` 추출 모듈을 강화한다.

- 오케스트레이션 로직은 `scripts/recipe-loop/loop.py`에 있고, 이 노트북은 그것을 호출한다.
- 무거운 작업(추출/채점)은 `scripts/recipe-loop/*.mjs`를 subprocess로 재사용한다.
- **격리**: validation은 집계 점수만, holdout은 루프 중 미채점. 모듈에 정답 하드코딩 시 확정검증 FAIL.
- 기본은 실행 안 함. 마지막 셀에서 `RUN_LOOP = True`로 바꿔야 실제 루프가 돈다.


## 1. 환경 점검 & 모듈 로드

In [ ]:
import sys, subprocess, shutil
from pathlib import Path

def find_root(p=None):
    cur = Path(p or Path.cwd()).resolve()
    for c in (cur, *cur.parents):
        if (c / "package.json").exists() and (c / ".git").exists():
            return c
    raise RuntimeError("프로젝트 루트를 찾지 못함")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "scripts" / "recipe-loop"))
import os; os.chdir(ROOT)

for tool in ["claude", "codex", "node"]:
    print(f"{tool}: {shutil.which(tool) or 'NOT FOUND'}")

import loop
from loop import LoopConfig, run_loop, run_iteration, grade_summaries, decide, check_no_hardcoded_answers, weak_train_cases_text, fmt_det, fmt_ai
print("loop 모듈 로드 완료 · 프로젝트:", ROOT)


## 2. 설정 & 기준선 확인
임계값은 M3 기준선(재료F1 0.899 / 분량 0.787 / 단계 0.813) 위로 잡아 개선 목표를 만든다.

In [ ]:
cfg = LoopConfig()
print("MAX_ITER:", cfg.max_iter, "| 모델:", cfg.model)
print("판정 임계값: 재료F1 >=", cfg.det_f1_min, ", 분량 >=", cfg.det_amount_min,
      ", 단계 >=", cfg.det_step_min, ", 레시피개수 >=", cfg.det_recipe_count_min,
      ", AI case >=", cfg.ai_case_min, ", AI avg >=", cfg.ai_avg_min)

base = grade_summaries(cfg, "baseline")
print("\n[기준선 baseline]")
print("train:", fmt_det(base["det"].get("aggregate", {})))
print("validation:", fmt_det(base["val"].get("aggregate", {})))


## 3. 배선 스모크 (쿼터/구현 없이)
하드코딩 점검 → baseline 채점 집계 → 판정 → 약점 케이스 추출 경로만 확인한다.

In [ ]:
ok, hits = check_no_hardcoded_answers()
print("하드코딩 점검:", "OK" if ok else f"FAIL {hits}")
decision = decide(cfg, base, ok)
print("판정 checks:", decision["checks"])
print("\n약점 케이스(계획/진단 입력):")
print(weak_train_cases_text(cfg, "baseline"))


## 4. 한 ITER 실행 (선택)
`RUN_ONE_ITER = True`로 바꾸면 ITER 1회만 실행한다(계획→구현→검증→채점→판정→진단).
Gemini 쿼터와 Codex 호출이 필요하다. 프롬프트가 바뀌면 train/validation 추출이 재실행된다(쿼터 소모).

In [ ]:
RUN_ONE_ITER = False

if RUN_ONE_ITER:
    from datetime import datetime
    run_dir = loop.RUN_ROOT / ("manual-" + datetime.now(loop.KST).strftime("%Y%m%d-%H%M%S"))
    run_dir.mkdir(parents=True, exist_ok=True)
    result = run_iteration(cfg, run_dir, 1, feedback="")
    print("\nITER 결과 passed:", result["passed"])
else:
    print("RUN_ONE_ITER=False → 실행 안 함")


## 5. 전체 루프 실행 (선택)
`RUN_LOOP = True`로 바꿔야 MAX_ITER까지 실제 루프가 돈다.

In [ ]:
RUN_LOOP = False

if RUN_LOOP:
    import json
    result = run_loop(cfg)
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("RUN_LOOP=False → 실행 안 함. True로 바꾸면 루프가 돈다.")
